# GymRec: train the exercise classifier on Google Colab

Dataset: [Gym Exercise DataSet (68K frames, 22 classes)](https://www.kaggle.com/datasets/amanyadav2030a/processed-train-test-exercise-68k)
by Aman Yadav (CC0). Each video is a folder of JPEG frames at about 5 fps.

Pipeline: **frames → YOLO-pose keypoints → 16-frame windows → train RF / XGBoost / LSTM / GRU / CNN →
pick the best on validation → test → `gymrec_model_<id>.zip`**. Install that zip on the website with one command.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU.**
2. **Kaggle API key:** kaggle.com → Settings → API → *Create New Token*. This downloads `kaggle.json`.
   In Colab, open the 🔑 **Secrets** panel (left sidebar) and add two secrets, both with notebook access on:
   `KAGGLE_USERNAME` = the "username" field from that file, and `KAGGLE_KEY` = the "key" field.
3. Run the cells in order. Outputs are saved to Google Drive (`MyDrive/gymrec`). If Colab disconnects,
   reconnect and run everything again: finished work is skipped.

In [ ]:
# 1. Check the GPU and connect Google Drive (where all outputs are kept)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU: Runtime > Change runtime type > T4 GPU"
from google.colab import drive
drive.mount("/content/drive")
WORK_DIR = "/content/drive/MyDrive/gymrec/work"

In [ ]:
# 2. Get the code (the training scripts must be pushed to GitHub)
%cd /content
!test -d Gym_Exercise || git clone -q https://github.com/Sarthak-D96/Gym_Exercise.git
%cd /content/Gym_Exercise
!git pull -q
!pip install -q ultralytics kagglehub
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.is_available())

In [ ]:
# 3. Download the dataset from Kaggle (~19 GB; about 5–15 minutes)
import os
from google.colab import userdata

for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        raise SystemExit(f"Add the Colab secret {k} (see the instructions at the top) and allow notebook access.")

import kagglehub
DATASET_ROOT = kagglehub.dataset_download("amanyadav2030a/processed-train-test-exercise-68k")
print("Downloaded to", DATASET_ROOT)

In [ ]:
# 4. Find the train / test folders (the ones holding one sub-folder per exercise)
from pathlib import Path

def find_dir(name):
    hits = sorted(p for p in Path(DATASET_ROOT).rglob(name) if p.is_dir())
    # if the name is nested (processed-train/processed-train), take the deepest one
    return max(hits, key=lambda p: len(p.parts)) if hits else None

TRAIN_DIR, TEST_DIR = find_dir("processed-train"), find_dir("processed-test")
print("train:", TRAIN_DIR)
print("test: ", TEST_DIR)
if TRAIN_DIR is None:
    !find "$DATASET_ROOT" -maxdepth 3 -type d | head -40
    raise SystemExit("processed-train not found: set TRAIN_DIR by hand from the listing above.")

# a few video folder names: do clips from one source video share a prefix? (see GROUP_REGEX below)
classes = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
print(len(classes), "classes:", classes)
print("example video folders:", sorted(p.name for p in (TRAIN_DIR / classes[0]).iterdir())[:12])

## 5. Configure

Defaults come from `training/gym_config.py`. Anything set here overrides them for this run.

* **`CLASSES`** — the exercises to train on (slugs from `config.EXERCISES`; the 22 dataset classes are all listed there).
  The website will show exactly these after installing the model.
* **`MODELS`** — which model types to compare. The best one on the **validation** set is exported.
* **`GROUP_REGEX`** — if the folder names above show that several clips come from one source video
  (e.g. `vid-squat_12_1`, `vid-squat_12_2`), set a regex whose group 1 is the source id, so those clips
  stay in the same split. Leave `None` if the names give no hint.

In [ ]:
import sys; sys.path.insert(0, "/content/Gym_Exercise/training")
import run_all
config = run_all.config   # training/gym_config.py; values set below override it

config.CLASSES = ["squat", "push_up", "shoulder_press", "bicep_curl", "bench_press", "pull_up"]
# all 22:  config.CLASSES = [c for c in config.EXERCISES if c not in ("lunge", "jumping_jack")]

config.DATA_DIRS = [str(TRAIN_DIR)]
config.TEST_DIRS = [str(TEST_DIR)] if TEST_DIR else []   # empty = split train 70/15/15 instead
config.WORK_DIR = WORK_DIR

# this dataset is frames already sampled at ~5 fps: use every frame
config.FRAMES_FPS = 5
config.FPS = 5          # the website should send ~5 fps too (FRAME_INTERVAL_MS = 200 in static/js/live.js)

config.TWO_STAGE = False   # one model for all classes (colab_two_stage.ipynb: router + specialists)
config.MODELS = ["rf", "xgb", "lstm", "gru", "cnn"]
config.SELECT_BY = "macro_f1"
# config.FORCE_MODEL = "lstm"      # ship this model instead of the best one
# config.GROUP_REGEX = r"^(.*)_\d+$"

In [ ]:
# 6. Check which dataset folders feed which class (every selected class should have videos)
counts = run_all.preview()

In [ ]:
# 7. Run everything: extract keypoints -> build dataset -> train all models -> pick the best and export
run_all.main()

# Re-run only some steps, e.g. after changing MODELS or DL settings:
#   run_all.main(["train", "select"])
# After changing CLASSES, run everything again (already-extracted videos are skipped):
#   run_all.main()

In [ ]:
# 8. Results and download
from pathlib import Path
from google.colab import files

latest = sorted(Path(WORK_DIR).glob("gymrec_model_*.zip"), key=lambda p: p.stat().st_mtime)[-1]
print(open(latest.with_suffix("") / "results.md").read())
print("Model zip:", latest, "(also kept in Google Drive)")
files.download(str(latest))

## 9. Use it on the website (on your laptop)

```bash
cd ~/Desktop/Gym_Exercise
pip install -r requirements.txt            # torch + ultralytics (+ scikit-learn / xgboost if a tree model won)
python manage.py install_model ~/Downloads/gymrec_model_<id>.zip
python manage.py runserver
```

`install_model` unpacks the zip into `ml_models/`, adds any new exercises, activates the model and the
pose weights, and shows only the trained exercises. For the journal, copy `results.md` (printed above)
and `summary.json` (in `MyDrive/gymrec/work/prepared/`) into Phases 3–4.